# Training on images and spectra together

This notebook trains a small PyTorch model on a multimodal HATS catalog, where every row holds both
an image and a spectrum of the same galaxy. The catalog is a crossmatch of Legacy Survey DR10 images
with DESI EDR spectra, hosted on Hugging Face.

The model is not meant to do real science. It shows how the different column types of a catalog
reach a model:

- **Images** are fixed-shape tensor columns (`tensor[float, (160, 160)]`), one per band.
- **Spectra** are a nested column, with one row of `flux`, `lambda`, `ivar`, ... per object.

Data is streamed with `CatalogDataset`, and several PyTorch workers load batches in parallel while
the model trains.

This notebook is pre-executed because it reads data over the network.

In [1]:
import lsdb
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F

from lsdb_ml import CatalogDataLoader, CatalogDataset

## Load the catalog

We open only the columns we need, the `g` and `r` band images and the spectra, and keep the first
32 of its 548 partitions so the notebook runs in a few minutes.

In [2]:
catalog = lsdb.open_catalog(
    "hf://datasets/brantd/multimodal_hats_example",
    columns=["image_flux_g", "image_flux_r", "spectrum"],
)
catalog = catalog.partitions[:32]
catalog

,image_flux_g,image_flux_r,ra_mmu_legacysurvey_dr10_south_21,dec_mmu_legacysurvey_dr10_south_21,spectrum
npartitions=32,,,,,
"Order: 7, Pixel: 41472","tensor[float, (160, 160)]","tensor[float, (160, 160)]",double[pyarrow],double[pyarrow],"nested<flux: [float], ivar: [float], lsf_sigma..."
...,...,...,...,...,...
"Order: 7, Pixel: 41602",...,...,...,...,...
"Order: 7, Pixel: 41603",...,...,...,...,...


## Turn rows into tensors

The default `collate_catalog` handles flat and nested numeric columns, but not tensor columns, so
we write a `collate_fn`. It receives a `NestedFrame` with one batch of rows and returns tensors:

- `to_stack()` turns a tensor column into one `(batch, 160, 160)` array. We stack the two bands
  into channels.
- Every DESI spectrum is sampled on the same wavelength grid, so the flat `flux` values reshape into
  a `(batch, n_wavelengths)` array.

In [3]:
def collate(frame):
    image = np.stack([frame["image_flux_g"].to_stack(), frame["image_flux_r"].to_stack()], axis=1)
    spectrum = frame["spectrum"].nest["flux"].to_numpy(dtype=np.float32).reshape(len(frame), -1)
    return {"image": torch.from_numpy(image), "spectrum": torch.tensor(spectrum)}


batch = collate(catalog.partitions[0].compute())
{name: tensor.shape for name, tensor in batch.items()}

Building task graph...


Computing Catalog:   0%|          | 0/1 [00:00<?, ?it/s]

{'image': torch.Size([7, 2, 160, 160]), 'spectrum': torch.Size([7, 7781])}

## Stream batches with several workers

With `num_workers > 0`, each PyTorch worker streams its own share of the partitions with dask's
synchronous scheduler, so no dask client is needed.

We start the workers with `forkserver`, because forking the notebook process directly (the default
on Linux) can hang. The dataset, including our `collate` function, is sent to each worker with
`cloudpickle`, so functions defined in a notebook work.

> **Faster worker startup.** Each worker imports torch and LSDB when it starts. To do this only
> once, preload them into the fork server:
> `ctx = multiprocessing.get_context("forkserver")`, `ctx.set_forkserver_preload(["torch", "lsdb"])`,
> and pass `multiprocessing_context=ctx`. **On macOS, preload only `torch`** when reading over HTTPS
> (e.g. `hf://`): with `lsdb`, `nested_pandas` or `pandas` preloaded, workers segfault on their
> first request, because the macOS certificate check (`truststore`) is not fork-safe. Here, reading
> from Hugging Face takes much longer than startup, so we skip it.

In [4]:
dataset = CatalogDataset(catalog, batch_size=64, seed=0, collate_fn=collate)
loader = CatalogDataLoader(dataset, num_workers=4, multiprocessing_context="forkserver")

## A two-tower model

Each modality gets its own small encoder: a 2D CNN for the images and a 1D CNN for the spectra.
Both map to a shared 64-dimensional embedding. A `BatchNorm` at the input of each encoder takes
care of the very different flux scales of the two.

We train it contrastively, as in CLIP or AstroCLIP: in a batch of `B` objects, the image of each
object should be closer to its own spectrum than to the other `B - 1` spectra. No labels are
needed.

In [5]:
class TwoTower(nn.Module):
    def __init__(self, dim=64):
        super().__init__()
        self.image = nn.Sequential(
            nn.BatchNorm2d(2),
            nn.Conv2d(2, 16, 5, stride=4),
            nn.ReLU(),
            nn.Conv2d(16, 32, 5, stride=4),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(32, dim),
        )
        self.spectrum = nn.Sequential(
            nn.BatchNorm1d(1),
            nn.Conv1d(1, 16, 9, stride=8),
            nn.ReLU(),
            nn.Conv1d(16, 32, 9, stride=8),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(32, dim),
        )

    def forward(self, image, spectrum):
        return F.normalize(self.image(image), dim=1), F.normalize(self.spectrum(spectrum[:, None]), dim=1)

## Train

`CatalogDataLoader` gives each epoch a new shuffle order. Besides the loss, we track how often an
image's nearest spectrum in the batch is its own; chance is about `1 / 64`.

In [6]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
)
model = TwoTower().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(3):
    losses, matches = [], []
    for batch in loader:
        image_emb, spectrum_emb = model(batch["image"].to(device), batch["spectrum"].to(device))
        logits = image_emb @ spectrum_emb.T / 0.1
        target = torch.arange(len(logits), device=device)
        loss = (F.cross_entropy(logits, target) + F.cross_entropy(logits.T, target)) / 2
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        matches.append((logits.argmax(1) == target).float().mean().item())
    print(f"epoch {epoch}: loss {np.mean(losses):.3f}, matched {np.mean(matches):.1%}")

epoch 0: loss 3.849, matched 4.5%


epoch 1: loss 3.635, matched 5.7%


epoch 2: loss 3.509, matched 6.5%
